# Voting regressor
Base regressors against a voting regressor on the Boston housing data, then weights, then one algorithm with
different depths. The interactive curve demo is `app.py` (`python app.py`, then open http://127.0.0.1:8050).

Conditions: folds are shuffled (the rows are stored town by town), every score is 10-fold cross-validation
repeated 5 times with different shuffles, and SVR gets standardized features (it measures distances).

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, RepeatedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

cv = RepeatedKFold(n_splits=10, n_repeats=5, random_state=0)   # shuffled folds, 5 different shuffles

## 1. The data
Boston housing: 506 districts, 13 inputs, median house price MEDV (thousands of dollars).
`load_boston` was removed from scikit-learn in 1.2; the same table is in data/boston.csv (OpenML dataset 531).

In [2]:
boston = pd.read_csv("data/boston.csv")
X = boston.drop(columns="MEDV")
y = boston["MEDV"]
print(X.shape, y.shape)

(506, 13) (506,)


## 2. Each base model, shuffled 10-fold cross-validation (5 repeats), R²

In [3]:
lr = LinearRegression()
dt = DecisionTreeRegressor(random_state=0)
svr = make_pipeline(StandardScaler(), SVR())
estimators = [("lr", lr), ("dt", dt), ("svr", svr)]

for name, model in estimators:
    scores = cross_val_score(model, X, y, scoring="r2", cv=cv)
    print(name, np.round(np.mean(scores), 3))

lr 0.712


dt 0.734


svr 0.668


## 3. The voting regressor: the mean of the three predictions

In [4]:
vr = VotingRegressor(estimators)
scores = cross_val_score(vr, X, y, scoring="r2", cv=cv)
print("voting regressor", np.round(np.mean(scores), 3))

voting regressor 0.809


In [5]:
# n_jobs=-1 on small data: starting parallel workers costs more than it saves (median of 5 fits)
import time
for jobs in [None, -1]:
    t = []
    for _ in range(5):
        start = time.perf_counter(); VotingRegressor(estimators, n_jobs=jobs).fit(X, y); t.append(time.perf_counter() - start)
    print("n_jobs =", jobs, f"{np.median(t):.2f} s")

n_jobs = None 0.10 s


n_jobs = -1 5.85 s


## 4. Weights from 1 to 3 for each model (27 combinations)

In [6]:
rows = []
for i in range(1, 4):
    for j in range(1, 4):
        for k in range(1, 4):
            vr = VotingRegressor(estimators, weights=[i, j, k])
            rows.append((i, j, k, np.mean(cross_val_score(vr, X, y, scoring="r2", cv=cv))))
weights = pd.DataFrame(rows, columns=["lr", "dt", "svr", "r2"]).round(3).sort_values("r2", ascending=False)
weights.head(8)

,lr,dt,svr,r2
15,2,3,1,0.820
24,3,3,1,0.818
3,1,2,1,0.818
12,2,2,1,0.817
16,2,3,2,0.817
6,1,3,1,0.815
25,3,3,2,0.815
7,1,3,2,0.814


## 5. One algorithm, five depths

In [7]:
trees = [(f"dt{d}", DecisionTreeRegressor(max_depth=d, random_state=0)) for d in [1, 3, 5, 7, None]]
for name, model in trees:
    print(name, np.round(np.mean(cross_val_score(model, X, y, scoring="r2", cv=cv)), 3))
print("voting regressor", np.round(np.mean(cross_val_score(VotingRegressor(trees), X, y, scoring="r2", cv=cv)), 3))

dt1 0.35


dt3 0.673


dt5 0.735


dt7 0.744


dtNone 0.734


voting regressor 0.757


## 6. Why we shuffle: the same models with folds in order
The Boston rows are stored in order of town, so plain `cv=10` (no shuffling) tests each model on towns it
never saw, and every score drops. SVR here is the plain SVR on raw features, as in older code.

In [8]:
plain = [("lr", LinearRegression()), ("dt", DecisionTreeRegressor(random_state=0)), ("svr", SVR())]
for name, model in plain + [("voting", VotingRegressor(plain))]:
    print(name, "folds in order:", np.round(np.mean(cross_val_score(model, X, y, scoring="r2", cv=10)), 2))

lr folds in order: 0.2
dt folds in order: -0.07


svr folds in order: -0.41


voting folds in order: 0.43


## 7. The curve demo (same code as app.py)

In [9]:
from app import figure, run

results = run(["linear regression", "SVR", "decision tree"])
print([(n, round(r2, 2), round(mae, 2)) for n, _, r2, mae in results])
figure(results)

[('voting regressor', 0.52, 0.31), ('linear regression', 0.18, 0.54), ('SVR', 0.47, 0.23), ('decision tree', 0.38, 0.23)]


In [10]:
# Folds in order vs towns: the corrected data (Gilley and Pace 1996, StatLib) names each row's town
from sklearn.model_selection import KFold, GroupKFold
bc = pd.read_csv("data/boston_corrected.txt", sep="\t", skiprows=8, encoding="latin1")
town = bc["TOWN"].values
print("towns:", len(set(town)), " unbroken runs:", 1 + (town[1:] != town[:-1]).sum())
for name, cv in [("in order", KFold(10)), ("shuffled", KFold(10, shuffle=True, random_state=42))]:
    seen = sum(t in set(town[tr]) for tr, te in cv.split(X) for t in town[te])
    print(name, "- test rows whose town is in training:", seen, "of", len(X))
for name, model in estimators + [("voting", VotingRegressor(estimators))]:  # scaled SVR
    print(name, "GroupKFold by town:", round(cross_val_score(model, X, y, scoring="r2", cv=GroupKFold(10), groups=town).mean(), 2))

towns: 92  unbroken runs: 92


in order - test rows whose town is in training: 53 of 506


shuffled - test rows whose town is in training: 487 of 506


lr GroupKFold by town: 0.55


dt GroupKFold by town: 0.36


svr GroupKFold by town: 0.61


voting GroupKFold by town: 0.68


In [11]:
# The curve demo on 30 freshly drawn sine datasets (same recipe as app.py): average test R² per model.
# For squared error the mean of the models can never be worse than the average of the models (Krogh and Vedelsby 1995).
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
r2 = {}
for s in range(30):
    rng = np.random.RandomState(s)
    Xs = np.sort(5 * rng.rand(80, 1), axis=0); ys = np.sin(Xs).ravel(); ys[::5] += 3 * (0.5 - rng.rand(16))
    a, b, c_, d = train_test_split(Xs, ys, test_size=0.1, random_state=s)
    base = [("linear regression", LinearRegression()), ("SVR", SVR()), ("decision tree", DecisionTreeRegressor(max_depth=5, random_state=0))]
    for k in [1, 2, 3]:
        vr = VotingRegressor(base[:k]).fit(a, c_)
        r2.setdefault(f"vote of {k}", []).append(r2_score(d, vr.predict(b)))
    for n, m in base:
        r2.setdefault(n, []).append(r2_score(d, m.fit(a, c_).predict(b)))
print({k: round(np.mean(v), 2) for k, v in r2.items()})

{'vote of 1': np.float64(0.45), 'vote of 2': np.float64(0.69), 'vote of 3': np.float64(0.72), 'linear regression': np.float64(0.45), 'SVR': np.float64(0.77), 'decision tree': np.float64(0.53)}
